# **F1 World Championship (1950 - 2024) Data Exploration**

Formula 1 (a.k.a. F1 or Formula One) is the highest class of single-seater auto racing sanctioned by the Fédération Internationale de l'Automobile (FIA) and owned by the Formula One Group. The FIA Formula One World Championship has been one of the premier forms of racing around the world since its inaugural season in 1950. The dataset consists of all information on the Formula 1 races, drivers, constructors, qualifying, circuits, lap times, pit stops, championships from 1950 till the latest 2024 season.

I would like to explore 3 areas with the vast amount of data available :

<div style="background-color: rgba(255, 24, 1, 0.18); padding: 10px; border-radius: 5px; border-left: 5px solid #FF1801;">
  <h2 style="color: #600b00; margin: 0; padding: 0; line-height: 1.2;">🏎️ 1. Race Outcome Prediction (ML Model)</h2>
</div>
<br>

Predict a driver's finishing position before the race starts. Rich feature set available: grid position, qualifying times (Q1/Q2/Q3), constructor, circuit, driver historical performance, pit stop strategy patterns. Could also branch into predicting DNFs specifically using the status table.

**Sub-targets:**
* Podium probability classifier
* DNF likelihood predictor
* Points scored regression

<div style="background-color: rgba(220, 0, 0, 0.15); padding: 10px; border-radius: 5px; border-left: 5px solid #DC0000;">
  <h2 style="color: #580000; margin: 0; padding: 0; line-height: 1.2;">⏱️ 2. Pit Stop Strategy Analysis (EDA + Conclusions)</h2>
</div>
<br>

The pit_stops table has millisecond-level timing. Cross it with race results and constructor data. You can derive whether faster/fewer stops correlate with better outcomes, which constructors have consistently elite pit crews, and how circuit type affects optimal stop count. 70+ years of strategy evolution visible here.

**Sub-targets:**
* Pit stop speed vs final position correlation
* Constructor pit crew consistency over eras
* Undercut/overcut effectiveness by circuit type

<div style="background-color: rgba(139, 0, 0, 0.15); padding: 10px; border-radius: 5px; border-left: 5px solid #8B0000;">
  <h2 style="color: #4A0000; margin: 0; padding: 0; line-height: 1.2;">🏆 3. Driver & Constructor Era Dominance (EDA + Statistical)</h2>
</div>
<br>

Using standings, results, and points data across all seasons, quantify how dominant each champion era actually was. Schumacher's Ferrari era vs Hamilton's Mercedes vs Vettel's Red Bull. Normalize for field size, point systems, and DNF rates to make cross-era comparisons fair.

**Sub-targets:**
* Adjusted dominance score per champion season
* Constructor win-rate consistency index
* "What if" parity analysis — how competitive was each season?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

from pprint import pprint
import os

In [ ]:
circuits = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/circuits.csv')
races = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/races.csv')
results = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/results.csv')
drivers = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/drivers.csv')
constructors = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/constructors.csv')
status = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/status.csv')
lap_times = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/lap_times.csv')
pit_stops = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/pit_stops.csv')
qualifying = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/qualifying.csv')
driver_standings = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/driver_standings.csv')
constructor_standings = pd.read_csv('/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020/constructor_standings.csv')

In [ ]:
datasets = {
    "circuits": circuits,
    "races": races,
    "results": results,
    "drivers": drivers,
    "constructors": constructors,
    "status": status
}

for name, df in datasets.items():
    print("\n", name)
    print(df.shape)
    print(df.head())

<div style="background-color: rgba(255, 24, 1, 0.10); padding: 10px; border-radius: 5px; border-left: 5px solid #FF1801;">
  <h2 style="color: #600b00; margin: 0; padding: 0; line-height: 1.2;">🏎️ 1. Race Outcome Prediction (ML Model)</h2>
</div>
<br>

Predict a driver's finishing position before the race starts. Rich feature set available: grid position, qualifying times (Q1/Q2/Q3), constructor, circuit, driver historical performance, pit stop strategy patterns. Could also branch into predicting DNFs specifically using the status table.

**Sub-targets:**
* Podium probability classifier
* DNF likelihood predictor
* Points scored regression

### Race Outcome Prediction : Exploratory Data Analysis

In [ ]:
print("------- Races Information Dataset -------")
races.info()

print("\n ------- Drivers Information Dataset -------")
drivers.info()

print("\n ------- Constructors Information Dataset -------")
constructors.info()

print("\n ------- Results Information Dataset -------")
results.info()

print("\n ------- Race Qualifying Information Dataset -------")
qualifying.info()

print("\n -------  Pitstop Information Dataset -------")
pit_stops.info()

For predicting a driver's finishing position using the dataset we would need to perform analysis for highly relevant data points and columns such as :
* driver qualifying times (drivers may perform better depending on their quali times)
* driver grid position at the race (drivers are highly likely to finish at a certain position depending on grid pos)
* driver's constructor (teams are known to dominate due to competitive edge)
* driver performances on circuits (studying driver performance and dominance on circuits)

For this data exploration, we would need to create a dataset with columns required :
* driver id
* race id
* constructor id
* race quali times (q1, q2, q3)
* grid position

In [ ]:
race_outcome = (
    qualifying[['raceId', 'driverId', 'constructorId', 'position', 'q1', 'q2', 'q3']]
    .merge(
        results[['raceId', 'driverId', 'grid', 'positionOrder', 'statusId']],
        on=['raceId', 'driverId'],
        how='inner'
    )
    .merge(
        races[['raceId', 'circuitId', 'year']],
        on='raceId',
        how='left'
    )
    .merge(
        drivers[['driverId', 'forename', 'surname']],
        on='driverId',
        how='left'
    )
    .merge(
        constructors[['constructorId', 'name']],
        on='constructorId',
        how='left'
    )
)

# Create the clean combined name column
race_outcome['driver_name'] = race_outcome['forename'] + ' ' + race_outcome['surname']
race_outcome = race_outcome.drop(columns=['forename', 'surname'])

# Rename columns to maintain your clean schema
race_outcome = race_outcome.rename(columns={
    'raceId':        'race_id',
    'driverId':      'driver_id',
    'constructorId': 'constructor_id',
    'name':          'constructor',
    'position':      'quali_position',
    'q1':            'q1_time',
    'q2':            'q2_time',
    'q3':            'q3_time',
    'grid':          'grid_position',
    'positionOrder': 'finish_position',
    'statusId':      'status_id',
    'circuitId':     'circuit_id',
    'year':          'race_year'
})

race_outcome.info()

In [ ]:
race_outcome.head()

In [ ]:
race_outcome.isnull().sum()

In [ ]:
race_outcome['q2_time'] = race_outcome['q2_time'].fillna('00:00:00')
race_outcome['q3_time'] = race_outcome['q3_time'].fillna('00:00:00')

In [ ]:
race_outcome.isnull().sum()

In [ ]:
meaningful_cols = [
    'quali_position', 
    'grid_position', 
    'finish_position', 
    'status_id', 
    'race_year'
]

# Calculate the matrix using 'spearman' for rank-based data
corr_matrix = race_outcome[meaningful_cols].corr(method='spearman')


plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='Reds', 
    fmt=".2f", 
    square=True, 
    linewidths=0.5
)

plt.title('F1 Race Outcome Feature Correlation (Spearman Rank)', fontsize=14)
plt.show()

The correlation heatmap diagram showcases deep information regarding the race outcome. Starting `grid position` for the driver highly correlates with the `finish position`. The `quali position` for a driver also correlated very highly for the `finish position`. The `grid position` and `quali position` are very highly correlated as these are almost the same values except for drivers disqualification and other reasons.

In [ ]:
race_outcome['position_gain'] = race_outcome['grid_position'] - race_outcome['finish_position']
# positive gain means a better result than starting position, and negative means bad race day

race_outcome[['grid_position', 'finish_position', 'position_gain']].head()

In [ ]:
target_year = 2015

modern_era_gains = (
    race_outcome[race_outcome['race_year'] >= target_year]
    .groupby('driver_name')['position_gain']
    .mean()
    .sort_values(ascending=False)
)

print(f"Top 10 Drivers by Average Positions Gained (Since {target_year})")
print(modern_era_gains.head(10))

In [ ]:
# column for labelling DNF drivers
race_outcome['is_dnf'] = (race_outcome['status_id'] != 1).astype(int)

dnfs_per_race = (
    race_outcome.groupby(['race_year', 'race_id'])['is_dnf']
    .sum()
    .reset_index()
)

yearly_dnf_avg = (
    dnfs_per_race.groupby('race_year')['is_dnf']
    .mean()
    .reset_index()
)
yearly_dnf_avg.rename(columns={'is_dnf': 'avg_dnfs_per_race'}, inplace=True)

plt.figure(figsize=(15, 4))
sns.lineplot(
    x='race_year', 
    y='avg_dnfs_per_race', 
    data=yearly_dnf_avg, 
    color='#E10600',
    linewidth=2.5
)


plt.axvline(x=2000, color='gray', linestyle='--', alpha=0.7)
plt.text(2001, 11, 'Bulletproof Engine Era', color='black', fontsize=10)

plt.title('The Evolution of F1 Reliability: Average DNFs per Race (1950–2024)', fontsize=14)
plt.xlabel('Season (Race Year)', fontsize=12)
plt.ylabel('Average Number of DNFs per Race', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

As engines and teams started to employ better engineering principles and cars got better over time, the average DNFs per race decreased on average, leading to more uniform and data-filled races where almost all drivers compete and the podium can be achieved by almost any driver on the grid (due to known unknowns during a race, different pit strategies and other factors)

In [ ]:
podium = pd.Series(race_outcome.finish_position <= 3)
race_outcome = pd.concat([race_outcome, podium.rename('is_podium')], axis=1)

race_outcome.head(20)

In [ ]:
race_outcome.info()

In [ ]:
# standardising missing vlaues

time_cols = ['q1_time', 'q2_time', 'q3_time']
for col in time_cols:
    race_outcome[col] = race_outcome[col].replace(r'\N', np.nan)

race_outcome.head(15)

In [ ]:
# converting the text strings for quali times into a continuos time variable, better for ml model understanding

race_outcome = race_outcome.copy()

# 2. Define standard time conversion helper
def time_to_seconds(time_str):
    if pd.isna(time_str) or not isinstance(time_str, str) or time_str in [r'\N', '00:00:00']:
        return np.nan
    try:
        if ':' in time_str:
            minutes, seconds = time_str.split(':')
            return int(minutes) * 60 + float(seconds)
        return float(time_str)
    except ValueError:
        return np.nan

# 3. Convert all qualifying strings to numerical float seconds
for col in ['q1_time', 'q2_time', 'q3_time']:
    race_outcome[col] = race_outcome[col].apply(time_to_seconds)

print("Qualifying columns converted successfully to float seconds!")

In [ ]:
# 1. Create flags to identify if a car actually took part in Q2 or Q3
race_outcome['q2_participated'] = race_outcome['q2_time'].notna().astype(int)
race_outcome['q3_participated'] = race_outcome['q3_time'].notna().astype(int)

# 2. Group by race_id and fill nulls with that specific weekend's median pace
for col in ['q2_time', 'q3_time']:
    race_outcome[col] = race_outcome.groupby('race_id')[col].transform(lambda x: x.fillna(x.median()))

# 3. Safe fallback for absolute corner cases where an entire historic session wasn't tracked
race_outcome['q1_time'] = race_outcome['q1_time'].fillna(race_outcome['q1_time'].median())
race_outcome['q2_time'] = race_outcome['q2_time'].fillna(race_outcome['q1_time'])
race_outcome['q3_time'] = race_outcome['q3_time'].fillna(race_outcome['q2_time'])

print("Session elimination features built. Missing times safely filled with weekend medians.")

In [ ]:
# 1. Ensure absolute column deduplication and establish target layout
race_outcome = race_outcome.loc[:, ~race_outcome.columns.duplicated()]
if 'is_podium' in race_outcome.columns:
    race_outcome.drop(columns=['is_podium'], inplace=True)

race_outcome['is_podium'] = (race_outcome['finish_position'] <= 3).astype(int)

# 2. Enforce strict chronological order before calculating rolling context
race_outcome = race_outcome.sort_values(by=['race_year', 'race_id']).reset_index(drop=True)

# 3. Calculate historical form (shifted by 1 to represent data prior to Sunday lights out)
race_outcome['driver_avg_finish_last_5'] = (
    race_outcome.groupby('driver_id')['finish_position']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

race_outcome['constructor_podium_rate_last_10'] = (
    race_outcome.groupby('constructor_id')['is_podium']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

# 4. Fill naive career boundaries (e.g., a driver's first career race gets historical mean metrics)
race_outcome['driver_avg_finish_last_5'] = race_outcome['driver_avg_finish_last_5'].fillna(race_outcome['finish_position'].mean())
race_outcome['constructor_podium_rate_last_10'] = race_outcome['constructor_podium_rate_last_10'].fillna(0)

# 5. Extract normalized margins (distance back from pole position in seconds)
race_outcome['q1_pole_time'] = race_outcome.groupby('race_id')['q1_time'].transform('min')
race_outcome['q1_delta_seconds'] = race_outcome['q1_time'] - race_outcome['q1_pole_time']
race_outcome.drop(columns=['q1_pole_time'], inplace=True)

print("Rolling context and qualifying delta features generated cleanly!")

## ML Model Development / Prediction

In [ ]:
# 1. Define distinct features (X) and target array (y)
feature_cols = [
    'grid_position', 'quali_position', 'q1_time', 'q2_time', 'q3_time',
    'q2_participated', 'q3_participated', 'driver_avg_finish_last_5', 
    'constructor_podium_rate_last_10', 'q1_delta_seconds', 'circuit_id'
]
target_col = 'is_podium'

# 2. Separate into historical learning, regulation transitions, and current seasons
train_mask = race_outcome['race_year'] < 2022
val_mask = race_outcome['race_year'].isin([2022, 2023])
test_mask = race_outcome['race_year'] == 2024

X_train, y_train = race_outcome.loc[train_mask, feature_cols], race_outcome.loc[train_mask, target_col]
X_val, y_val = race_outcome.loc[val_mask, feature_cols], race_outcome.loc[val_mask, target_col]
X_test, y_test = race_outcome.loc[test_mask, feature_cols], race_outcome.loc[test_mask, target_col]

print(f"Data partitioning complete:")
print(f" - Train Shape (Pre-2022): {X_train.shape}")
print(f" - Validation Shape (2022-2023): {X_val.shape}")
print(f" - Test Shape (2024 Benchmark): {X_test.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the Random Forest baseline model
baseline_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)

# 2. Fit the model on the historical training set
baseline_model.fit(X_train, y_train)

print("Baseline Random Forest model trained successfully!")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# 1. Predict probabilities and hard classes for the validation set
val_probs = baseline_model.predict_proba(X_val)[:, 1]
val_preds = baseline_model.predict(X_val)

# 2. Print evaluation metrics
print("--- Validation Set Performance (2022-2023 Era) ---")
print(f"ROC-AUC Score: {roc_auc_score(y_val, val_probs):.4f}\n")
print(classification_report(y_val, val_preds, target_names=["No Podium", "Podium"]))

In [ ]:
# 1. Map features to their respective importance scores
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': baseline_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# 2. Display the sorted leaderboard
print("--- Feature Importance Leaderboard ---")
print(importances.to_string(index=False))

In [ ]:
# Create a clean dataframe specifically for observing validation race weekends
val_analysis = race_outcome.loc[val_mask].copy()
val_analysis['predicted_podium_prob'] = val_probs

print("Validation analysis table successfully mapped with predicted probabilities!")

In [ ]:
# Rank drivers within each race based on their predicted probability
val_analysis['predicted_rank'] = val_analysis.groupby('race_id')['predicted_podium_prob'].rank(ascending=False, method='first')

# Flag our top 3 choices for each race weekend
val_analysis['pred_is_podium'] = (val_analysis['predicted_rank'] <= 3).astype(int)

In [ ]:
# Calculate how many of our predicted top-3 finishers were actually true podium finishers
correct_podium_predictions = val_analysis[val_analysis['pred_is_podium'] == 1]['is_podium'].sum()
total_podium_slots_predicted = val_analysis['pred_is_podium'].sum()

true_race_accuracy = (correct_podium_predictions / total_podium_slots_predicted) * 100
print(f"--- Real-World Race Prediction Performance ---")
print(f"True Podium Prediction Accuracy: {true_race_accuracy:.2f}%")

In [ ]:
sample_race_id = val_analysis['race_id'].iloc[0] # Grabs the first race in the validation set
sample_race = val_analysis[val_analysis['race_id'] == sample_race_id].sort_values(by='predicted_rank')

print(f"\n--- Sample Race Simulation (Race ID: {sample_race_id}) ---")
print(sample_race[['driver_name', 'constructor', 'grid_position', 'finish_position', 'predicted_podium_prob', 'pred_is_podium']].head(6).to_string(index=False))

### Evaluating the Final 2024 Benchmark

In [ ]:
# 1. Generate out-of-sample probabilities for the 2024 season
test_analysis = race_outcome.loc[test_mask].copy()
test_analysis['predicted_podium_prob'] = baseline_model.predict_proba(X_test)[:, 1]

# 2. Group by each 2024 Grand Prix weekend and isolate the predicted top 3
test_analysis['predicted_rank'] = test_analysis.groupby('race_id')['predicted_podium_prob'].rank(ascending=False, method='first')
test_analysis['pred_is_podium'] = (test_analysis['predicted_rank'] <= 3).astype(int)

# 3. Calculate final out-of-sample accuracy
correct_test_predictions = test_analysis[test_analysis['pred_is_podium'] == 1]['is_podium'].sum()
total_test_slots = test_analysis['pred_is_podium'].sum()
test_accuracy = (correct_test_predictions / total_test_slots) * 100

print(f"--- Final 2024 Season Test Performance ---")
print(f"Out-of-Sample Podium Prediction Accuracy: {test_accuracy:.2f}%")

In [ ]:
# Extract predictions that failed in reality
missed_predictions = test_analysis[(test_analysis['pred_is_podium'] == 1) & (test_analysis['is_podium'] == 0)]

print("\n--- Model Misses / False Alarms in 2024 ---")
print(missed_predictions[['race_year', 'driver_name', 'constructor', 'grid_position', 'finish_position', 'predicted_podium_prob']].to_string(index=False))

## The 2024 Error Analysis Breakdown
1. The Red Bull Mid-Season Performance Cliff (Pérez)
The model predicted Sergio Pérez to hit the podium in both China (predicted probability: 0.86) and Miami (0.53), but he finished 3rd and 4th.

The Glitch: In early 2024, Red Bull was still massively dominant, so Pérez’s `constructor_podium_rate_last_10` was incredibly high. The model couldn't know that Red Bull's upgrade package would introduce handling issues that caused his individual performance to drop, while McLaren and Ferrari caught up in developmental pace.

2. The Lando Norris Pit-Stop Windfall (Miami)
The model predicted Lando Norris to miss the podium (predicted prob: 0.29, finishing outside its top 3 choices), but he won the race.

The Glitch: Norris qualified 5th. In a normal race, passing a dominant Red Bull and Ferrari from 5th is statistically unlikely. However, a mid-race Safety Car came out at the exact perfect moment, allowing Norris a "free" pit stop to jump the entire field. This is pure race variance that a static pre-race model can never foresee.

3. The Sudden Mechanical DNF (Verstappen in Australia)
The model slapped a massive 0.96 probability on Max Verstappen in Australia, but he finished 19th.

The Glitch: Verstappen qualified on Pole (`grid_position = 1`). On lap 3, his right-rear brake caught fire, forcing a retirement. Because your model evaluates the race before the lights go out, it did its job perfectly—Verstappen was the objective favorite, but a mechanical failure is an unpredictable wildcard.

<div style="background-color: rgba(220, 0, 0, 0.15); padding: 10px; border-radius: 5px; border-left: 5px solid #DC0000;">
  <h2 style="color: #580000; margin: 0; padding: 0; line-height: 1.2;">⏱️ 2. Pit Stop Strategy Analysis (EDA + Conclusions)</h2>
</div>
<br>

The `pit_stops` table has millisecond-level timing. Cross it with race results and constructor data. You can derive whether faster/fewer stops correlate with better outcomes, which constructors have consistently elite pit crews, and how circuit type affects optimal stop count. 70+ years of strategy evolution visible here.

**Sub-targets:**
* Pit stop speed vs final position correlation
* Constructor pit crew consistency over eras
* Undercut/overcut effectiveness by circuit type


In [ ]:
pit_analysis = (
    pit_stops
    .merge(
        results[['raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'statusId']],
        on=['raceId', 'driverId'],
        how='inner'
    )
    .merge(
        races[['raceId', 'circuitId', 'year', 'name', 'round']],
        on='raceId',
        how='left'
    )
    .merge(
        drivers[['driverId', 'forename', 'surname']],
        on='driverId',
        how='left'
    )
    .merge(
        constructors[['constructorId', 'name']],
        on='constructorId',
        how='left',
        suffixes=('_race', '_team')
    )
)

pit_analysis['driver_full_name'] = pit_analysis['forename'] + ' ' + pit_analysis['surname']

column_order = [
    'year', 'name_race', 'round', 'driver_full_name', 'name_team',
    'stop', 'lap', 'time', 'duration', 'milliseconds', 
    'grid', 'positionOrder', 'statusId'
]
pit_analysis = pit_analysis[column_order]

pit_analysis['pit_seconds'] = pit_analysis['milliseconds'] / 1000.0

pit_analysis.head()

In [ ]:
normal_stops = pit_analysis[pit_analysis['pit_seconds'] < 45].copy()
corr_cols = ['year', 'stop', 'lap', 'pit_seconds', 'grid', 'positionOrder']
corr_matrix = normal_stops[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='Reds', fmt=".2f", linewidths=0.5)
plt.title('Formula 1 Pit Strategy & Race Outcome Correlation Matrix')
plt.show()